# Worked Examples — Advanced

Six explained solutions covering bounded async work, descriptors, rate limiting, property tests, transactions and safe logging. None of them are in the graded practice.

Each problem goes **plan → solution → why it works → common mistake → your turn**. Cover the solution, write your own version first, then compare.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "worked_examples"))
    sys.path.insert(0, os.getcwd())

## 1. Bounded fetches with per-task timeouts

Fetch many URLs (simulated) with at most 3 in flight. Give each a 0.1 s timeout and return a summary of successes and timeouts.

**Plan**

Use a semaphore for the bound, `asyncio.timeout` per task, and a TaskGroup so nothing outlives the function.

In [ ]:
import sys; sys.path.insert(0, "../notebooks")
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio

async def fake_fetch(url):
    await asyncio.sleep(0.5 if "slow" in url else 0.01)
    return f"<html {url}>"

async def fetch_all(urls, limit=3, timeout=0.1):
    semaphore = asyncio.Semaphore(limit)
    results = {}

    async def one(url):
        async with semaphore:
            try:
                async with asyncio.timeout(timeout):
                    results[url] = ("ok", await fake_fetch(url))
            except TimeoutError:
                results[url] = ("timeout", None)

    async with asyncio.TaskGroup() as group:
        for url in urls:
            group.create_task(one(url))
    return {url: results[url][0] for url in urls}

summary = run(fetch_all(["a", "slow-b", "c", "d"]))
print(summary)
assert summary == {"a": "ok", "slow-b": "timeout", "c": "ok", "d": "ok"}

**Why it works:** The timeout sits *inside* the semaphore, so time spent waiting for a slot doesn't count against a request. The TaskGroup waits for every task.

**Common mistake:** Catching `asyncio.CancelledError` instead of `TimeoutError`, which swallows real cancellation of the whole job.

**Your turn:** Retry each timed-out URL once before giving up. 

## 2. A reusable bounded-number descriptor

Create `Bounded(low, high)` so class attributes reject out-of-range values with a clear message naming the attribute.

**Plan**

`__set_name__` learns the attribute name, `__set__` validates, and `__get__` reads from the instance `__dict__`.

In [ ]:
class Bounded:
    def __init__(self, low, high):
        self.low, self.high = low, high

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not self.low <= value <= self.high:
            raise ValueError(f"{self.name} must be in [{self.low}, {self.high}], got {value}")
        instance.__dict__[self.name] = value

class Sensor:
    humidity = Bounded(0, 100)
    celsius = Bounded(-50, 60)

    def __init__(self, humidity, celsius):
        self.humidity, self.celsius = humidity, celsius

s = Sensor(40, 21.5)
try:
    s.humidity = 140
except ValueError as error:
    print(error)
assert (s.humidity, s.celsius) == (40, 21.5)

**Why it works:** The descriptor is a data descriptor (it defines `__set__`), so it takes priority over the instance dict. Storing the value under the same name in `__dict__` is safe.

**Common mistake:** Storing the value on the descriptor (`self.value = value`), which makes every Sensor share one humidity.

**Your turn:** Make the error message include the class name, e.g. `Sensor.humidity`.

## 3. Sliding-window rate limiter

Allow at most `n` events in any `window` seconds, using an injected clock.

**Plan**

Keep the timestamps of recent events in a deque. Drop the old ones, then allow the event if fewer than `n` remain.

In [ ]:
from collections import deque

class SlidingWindowLimiter:
    def __init__(self, n, window, clock):
        self.n, self.window, self.clock = n, window, clock
        self.events = deque()

    def allow(self):
        now = self.clock()
        while self.events and now - self.events[0] >= self.window:
            self.events.popleft()
        if len(self.events) < self.n:
            self.events.append(now)
            return True
        return False

now = [0.0]
limiter = SlidingWindowLimiter(n=2, window=1.0, clock=lambda: now[0])
decisions = []
for t in [0.0, 0.2, 0.5, 1.0, 1.1, 1.3]:
    now[0] = t
    decisions.append(limiter.allow())
print(decisions)
assert decisions == [True, True, False, True, False, True]

**Why it works:** A deque makes removing expired events from the left O(1). With the fake clock the behaviour is exact and repeatable.

**Common mistake:** Counting events in fixed one-second buckets, which allows up to 2n events across a bucket boundary.

**Your turn:** Add `retry_after()`, returning how long until the next event would be allowed.

## 4. Property test for a run-length encoder

Write `encode('aaab') -> [('a', 3), ('b', 1)]` and `decode`, and prove they round-trip on many random strings.

**Plan**

Use `itertools.groupby` for encoding. Then check the property `decode(encode(s)) == s` on 500 seeded random strings.

In [ ]:
import itertools
import random

def encode(text):
    return [(ch, len(list(group))) for ch, group in itertools.groupby(text)]

def decode(pairs):
    return "".join(ch * count for ch, count in pairs)

rng = random.Random(42)
for _ in range(500):
    s = "".join(rng.choice("ab ") for _ in range(rng.randint(0, 30)))
    assert decode(encode(s)) == s, s
    assert all(count > 0 for _, count in encode(s))
    assert all(a != b for (a, _), (b, _) in zip(encode(s), encode(s)[1:]))  # runs are maximal
print(encode("aaab  b"))

**Why it works:** The properties (round-trip, positive counts, no two equal neighbouring runs) define correctness better than a handful of examples. The seed makes any failure reproducible.

**Common mistake:** Testing only `'aaab'`. The empty string and strings with spaces are where bugs hide.

**Your turn:** Make the property test print the smallest failing string when it finds a bug.

## 5. Transactional outbox

When an order is saved, an `order.created` event must be stored too. Either both rows exist or neither does.

**Plan**

Write both rows in one transaction with `with conn:`. A failure in the second insert rolls back the first.

In [ ]:
import json
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE orders (id TEXT PRIMARY KEY, total REAL NOT NULL CHECK (total >= 0));
CREATE TABLE outbox (id INTEGER PRIMARY KEY, event TEXT NOT NULL, payload TEXT NOT NULL);
""")

def place_order(conn, order_id, total):
    with conn:
        conn.execute("INSERT INTO orders VALUES (?, ?)", (order_id, total))
        conn.execute("INSERT INTO outbox (event, payload) VALUES (?, ?)",
                     ("order.created", json.dumps({"id": order_id, "total": total})))

place_order(conn, "o1", 25.0)
for bad in [("o1", 5.0), ("o2", -1.0)]:      # duplicate id, negative total
    try:
        place_order(conn, *bad)
    except sqlite3.IntegrityError as error:
        print("rejected:", error)
counts = [conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("orders", "outbox")]
print(counts)
assert counts == [1, 1]

**Why it works:** The order and its event commit together, so a separate publisher can safely read the outbox and send events later (at least once).

**Common mistake:** Publishing to the message broker inside the transaction. If the commit then fails, you have announced an order that doesn't exist.

**Your turn:** Write `publish_pending(conn, send)` that sends outbox rows and deletes them only after `send` succeeds.

## 6. Redacting logs with a logging.Filter

Make sure no log record ever contains a value for `password=` or `token=`, whoever writes the log call.

**Plan**

Attach a `logging.Filter` to the handler. It rewrites the formatted message with a regex before output.

In [ ]:
import io
import logging
import re

SECRET = re.compile(r"(password|token)=\S+", re.IGNORECASE)

class RedactSecrets(logging.Filter):
    def filter(self, record):
        record.msg = SECRET.sub(r"\1=***", record.getMessage())
        record.args = ()
        return True

stream = io.StringIO()
handler = logging.StreamHandler(stream)
handler.addFilter(RedactSecrets())
log = logging.getLogger("worked.example")
log.addHandler(handler)
log.setLevel(logging.INFO)
log.propagate = False

log.info("login user=%s password=%s", "ana", "hunter2")
log.info("calling api TOKEN=abc123 status=%d", 200)
output = stream.getvalue()
print(output)
assert "hunter2" not in output and "abc123" not in output
assert "password=***" in output

**Why it works:** The filter runs on every record that reaches the handler. It formats the message first (`getMessage()`), so secrets passed as `%s` arguments are caught too.

**Common mistake:** Redacting only `record.msg` without the args, which misses `log.info('password=%s', secret)`.

**Your turn:** Also redact values in a dict passed via `extra={...}`.

**Next:** the [Advanced projects](../projects/) and [auto-graded practice](../../practice/README.md).